# Projeto: Atendente Virtual com RAG e Human-in-the-Loop

Neste notebook vamos combinar todos os conceitos da aula 3 para construir um **atendente virtual inteligente** da NovaTech Financeira. O sistema usa:

- **RAG** para consultar a base de conhecimento interna (produtos, taxas, políticas)
- **Busca na web** para informações externas e atualizadas
- **Human-in-the-Loop** para ações críticas (cancelamento, reembolso)
- **Memória** para manter o contexto da conversa

O agente responde perguntas sobre a empresa de forma autônoma, mas pede aprovação antes de executar ações irreversíveis.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Base de conhecimento

Setup do RAG: carregar documentos, dividir em chunks e indexar no vector store.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

loader = TextLoader("resources/base_conhecimento.md")
documentos = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documentos)

vectorstore = InMemoryVectorStore.from_documents(
    chunks,
    OpenAIEmbeddings(model="text-embedding-3-small")
)

print(f"Base de conhecimento pronta: {len(chunks)} chunks.")

Base de conhecimento pronta: 7 chunks.


## Ferramentas

O atendente tem cinco ferramentas:
- **consultar_base**: RAG sobre documentos internos
- **buscar_na_web**: informações externas
- **consultar_conta**: dados da conta do cliente
- **cancelar_conta**: ação crítica com aprovação
- **solicitar_reembolso**: ação crítica com aprovação

In [3]:
from langchain.tools import tool
from langgraph.types import interrupt
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def consultar_base(pergunta: str) -> str:
    """Consulta a base de conhecimento interna da NovaTech.
    Use para perguntas sobre produtos, taxas, politicas e procedimentos da empresa."""
    resultados = vectorstore.similarity_search(pergunta, k=3)
    contexto = "\n\n".join([doc.page_content for doc in resultados])
    return f"Informacoes da base interna:\n\n{contexto}"

@tool
def buscar_na_web(query: str) -> Dict[str, Any]:
    """Busca informacoes atualizadas na internet.
    Use para noticias, cotacoes e informacoes externas a empresa."""
    return tavily_client.search(query)

In [4]:
@tool
def consultar_conta() -> str:
    """Consulta informacoes da conta do cliente: saldo, fatura e status."""
    return (
        "Titular: Maria Silva\n"
        "Conta: 12345-6\n"
        "Saldo: R$ 5.230,00\n"
        "Fatura atual do cartao: R$ 1.850,00 (vencimento 15/04/2026)\n"
        "Cartao: NovaTech Plus\n"
        "Pontos acumulados: 12.350 pontos\n"
        "Status: Ativa"
    )

In [5]:
@tool
def cancelar_conta(motivo: str) -> str:
    """Cancela a conta do cliente. Acao irreversivel que requer aprovacao."""
    aprovacao = interrupt(
        f"APROVACAO NECESSARIA: Cancelamento de conta. Motivo: '{motivo}'. Aprovar? (sim/nao)"
    )
    if aprovacao == "sim":
        return f"Conta cancelada. Motivo: {motivo}. Protocolo: #2026-0417-001"
    return "Cancelamento nao aprovado pelo supervisor. Conta permanece ativa."

@tool
def solicitar_reembolso(valor: float, descricao: str) -> str:
    """Solicita reembolso de cobranca indevida no cartao. Requer aprovacao."""
    aprovacao = interrupt(
        f"APROVACAO NECESSARIA: Reembolso de R$ {valor:.2f} - {descricao}. Aprovar? (sim/nao)"
    )
    if aprovacao == "sim":
        return f"Reembolso de R$ {valor:.2f} aprovado. Credito na proxima fatura. Protocolo: #2026-0417-002"
    return "Reembolso nao aprovado."

## Atendente virtual

O agente coordenador com todas as ferramentas, memória e checkpointer para suportar HITL.

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

atendente = create_agent(
    model="gpt-4.1-mini",
    tools=[consultar_base, buscar_na_web, consultar_conta, cancelar_conta, solicitar_reembolso],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Voce e o atendente virtual da NovaTech Financeira. Seu nome e Nova.\n"
        "Diretrizes:\n"
        "- Para duvidas sobre produtos, taxas e politicas: consulte a base de conhecimento interna\n"
        "- Para informacoes de mercado ou externas: use a busca na web\n"
        "- Para dados da conta do cliente: use a ferramenta de consulta de conta\n"
        "- Para cancelamentos e reembolsos: use as ferramentas IMEDIATAMENTE quando o cliente solicitar. "
        "As ferramentas ja possuem mecanismo interno de aprovacao, entao NAO pergunte confirmacao antes de usa-las.\n"
        "- Seja cordial, objetivo e sempre indique de onde veio a informacao"
    )
)

config = {"configurable": {"thread_id": "atendimento-maria"}}

## Conversa: consulta sobre produtos

O primeiro turno é uma pergunta informacional. O agente deve consultar a base de conhecimento via RAG.

In [7]:
from langchain.messages import HumanMessage

resposta = atendente.invoke(
    {"messages": [HumanMessage(content="Oi! Quais tipos de cartao de credito voces oferecem?")]},
    config
)

print(resposta["messages"][-1].content)

A NovaTech Financeira oferece três modalidades de cartão de crédito:

1. NovaTech Essencial:
- Sem anuidade
- Limite inicial de R$ 500 a R$ 5.000
- Cashback de 0,5% em todas as compras
- Parcelamento em até 12x sem juros em lojas parceiras

2. NovaTech Plus:
- Anuidade de R$ 19,90/mês (isenta para gastos acima de R$ 3.000/mês)
- Limite de R$ 5.000 a R$ 25.000
- Cashback de 1,0% em todas as compras
- Sala VIP em aeroportos nacionais (2 acessos por ano)
- Parcelamento em até 18x sem juros em lojas parceiras

3. NovaTech Black:
- Anuidade de R$ 59,90/mês (isenta para gastos acima de R$ 8.000/mês)
- Limite de R$ 25.000 a R$ 100.000
- Cashback de 1,5% em todas as compras
- Sala VIP ilimitada em aeroportos nacionais e internacionais
- Seguro viagem internacional incluso
- Concierge 24 horas

Essas informações são da base interna da NovaTech. Posso ajudar com mais alguma coisa?


## Conversa: dados da conta

O cliente pede informações sobre sua conta. O agente usa a tool `consultar_conta`.

In [8]:
resposta = atendente.invoke(
    {"messages": [HumanMessage(content="Qual meu saldo e quantos pontos eu tenho?")]},
    config
)

print(resposta["messages"][-1].content)

Seu saldo atual é de R$ 5.230,00 e você possui 12.350 pontos acumulados no seu cartão NovaTech Plus. Posso ajudar com mais alguma informação?


## Conversa: pergunta com RAG + Web

Uma pergunta que exige informações internas e externas. O agente precisa consultar a base para saber o rendimento do CDB (110% do CDI) e a web para descobrir o CDI atual.

In [9]:
resposta = atendente.invoke(
    {"messages": [HumanMessage(content="O CDB de voces rende 110% do CDI. Qual e o CDI atual e quanto renderia R$ 10.000 em um ano?")]},
    config
)

print(resposta["messages"][-1].content)

O CDI atual está em aproximadamente 14,51% ao ano (conforme dados recentes do mercado).

Para calcular o rendimento de um CDB que rende 110% do CDI para um investimento de R$ 10.000 em um ano, usamos a fórmula de juros compostos:

Rendimento anual = Capital inicial * (1 + CDI * 1,10) ^ 1 - Capital inicial

Com CDI = 14,51% (0,1451), teríamos:

Rendimento anual = 10.000 * (1 + 0.1451 * 1.10) - 10.000 = 10.000 * 1.15961 - 10.000 = R$ 1.596,10

Portanto, ao final de um ano, seu investimento de R$ 10.000 renderia aproximadamente R$ 1.596,10, totalizando cerca de R$ 11.596,10.

Essas informações vieram da consulta à web sobre a taxa CDI atual e da base interna da NovaTech sobre os produtos. Posso ajudar com mais alguma coisa?


## Conversa: ação crítica com HITL

Agora a cliente decide cancelar a conta. O agente deve chamar a tool `cancelar_conta`, que contém `interrupt()` e vai pausar a execução aguardando aprovação.

In [10]:
resposta = atendente.invoke(
    {"messages": [HumanMessage(content="Quero cancelar minha conta. Motivo: vou migrar para outro banco.")]},
    config
)

print("Agente pausado para aprovação.")

Agente pausado para aprovação.


In [11]:
state = atendente.get_state(config)

print(f"Proximo passo: {state.next}")
for task in state.tasks:
    if task.interrupts:
        for interrupt_data in task.interrupts:
            print(f"Interrupcao: {interrupt_data.value}")

Proximo passo: ('tools',)
Interrupcao: APROVACAO NECESSARIA: Cancelamento de conta. Motivo: 'vou migrar para outro banco'. Aprovar? (sim/nao)


In [12]:
from langgraph.types import Command

resposta = atendente.invoke(
    Command(resume="sim"),
    config
)

print(resposta["messages"][-1].content)

Sua conta foi cancelada conforme solicitado, por motivo de migração para outro banco. Protocolo de atendimento: #2026-0417-001. Se precisar de mais alguma coisa, estou à disposição.


## Conversa: continuidade com memória

Após a aprovação do cancelamento, a conversa continua normalmente com o histórico preservado. O agente lembra de todos os turnos anteriores.

In [13]:
resposta = atendente.invoke(
    {"messages": [HumanMessage(content="Obrigada! Quanto eu tenho de pontos e como posso trocar por milhas?")]},
    config
)

print(resposta["messages"][-1].content)

Você possui 12.350 pontos acumulados no seu cartão NovaTech Plus. 

No programa NovaTech Rewards, a conversão para milhas aéreas é a seguinte:
- 1.000 pontos equivalem a 500 milhas aéreas
- O mínimo para resgate é 500 pontos
- Os pontos expiram em 24 meses após a data de acumulação

Se quiser, posso ajudar a iniciar a troca dos seus pontos por milhas aéreas. Deseja prosseguir?


Neste projeto combinamos **RAG**, **busca na web**, **Human-in-the-Loop** e **memória** em um único agente. O atendente virtual:

- Responde perguntas sobre a empresa consultando documentos internos via RAG
- Busca informações externas quando necessário (CDI, cotações)
- Acessa dados da conta do cliente
- Pausa e pede aprovação humana para ações críticas como cancelamento
- Mantém contexto ao longo de toda a conversa

Esse padrão arquitetural é a base de atendentes virtuais em produção, onde autonomia e controle humano coexistem de forma equilibrada.